# grid_100x100 四部屋 PIE シミュレーション

**前提**: UE Editor で `grid_100x100` を開き **PIE 実行中**。

## シナリオ

1. 領域 **(1,1)-(30,30)** を外周壁で囲み、**(10,10)** を柱として4部屋に分割
2. ドア幅 **90cm**（3マス）: SW↔SE, SW↔NW, SE↔NE（NEはSEからのみ）
3. **(20,20)** を実体ブロック化
4. SpotDog **(5,5)** → **(20,20)** → **(5,5)** 往復

ロジック: `grid_env_10k_four_rooms_pie.py` / `grid_env_10k_four_rooms_layout.py`

## 接続（重要）

- UnrealCV は **TCP 1 クライアントのみ**。複数 `Established` / `CloseWait` があると接続失敗します。
- PIE 再起動後: **Cell 3** で `release_ue_connection()` → **Cell 2** → **Cell 5**
- WSL: `ss -tnp state established '( dport = :9000 )'` で `python` が残っていれば Kernel 再起動または `kill`
- Windows: `Get-NetTCPConnection -LocalPort 9000` で **Listen（UE）+ Established 1 本** が正常

In [ ]:
import importlib
import sys
from pathlib import Path
from typing import Optional, Tuple

from simworld.communicator.communicator import Communicator
from simworld.communicator.unrealcv import UnrealCV


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_geh_dir = _root / "dev" / "grid_env_hri"
_g10k = _root / "dev" / "grid_env_10k"
for p in (_root, _geh_dir, _g10k):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

_prev_ucv: Optional[UnrealCV] = globals().get("ucv")
_prev_comm: Optional[Communicator] = globals().get("communicator")

import grid_env_10k as g10k
import grid_env_10k_four_rooms_pie as four_rooms

importlib.reload(g10k)
importlib.reload(four_rooms)

ucv: Optional[UnrealCV] = None
communicator: Optional[Communicator] = None

if _prev_ucv is not None:
    try:
        ucv, communicator = g10k.adopt_ue_session(_prev_ucv, _prev_comm)
    except (ConnectionError, Exception):
        try:
            _prev_ucv.disconnect()
        except Exception:
            pass
        print("[UE] prior session not reusable — will connect on ensure_connection()")


def ensure_connection(*, force_new: bool = False) -> Tuple[UnrealCV, Communicator]:
    global ucv, communicator
    ucv, communicator = four_rooms.ensure_single_ue_session(
        ucv=ucv, communicator=communicator, force_new=force_new
    )
    return ucv, communicator


def release_ue_connection() -> None:
    global ucv, communicator
    g10k.release_connection(ucv, communicator=communicator)
    ucv = None
    communicator = None


print(f"[Paths] root={_root}")

In [ ]:
# PIE 再起動後や接続エラー時のみ実行
# release_ue_connection()

In [ ]:
ROBOT_START = four_rooms.ROBOT_START_CELL
ROBOT_GOAL = four_rooms.ROBOT_GOAL_CELL
GOAL_DWELL_S = four_rooms.GOAL_DWELL_S
print(f"robot {ROBOT_START} -> {ROBOT_GOAL}, dwell={GOAL_DWELL_S}s")

In [ ]:
# PIE 再起動直後は force_new=True 推奨
ucv, communicator = ensure_connection(force_new=True)
result = four_rooms.run_four_rooms_scenario(
    robot_start_cell=ROBOT_START,
    robot_goal_cell=ROBOT_GOAL,
    goal_dwell_s=GOAL_DWELL_S,
    skip_robot_probe=True,
    ucv=ucv,
    communicator=communicator,
)
result

In [ ]:
success = four_rooms.four_rooms_success(result)
print(
    f"SUCCESS={success}, return_dist={result.return_dist_cm:.1f} cm, "
    f"outbound={result.outbound_arrived}, return={result.return_arrived}"
)